# ReducedSymmetryP2P

Compares the normal P2P packing for each selected CPU or CUDA backend against the explicit `use_reduced_symmetry_p2p` Tensor6 dictionary plan. It reports stored tensors, P2P-plan memory, field agreement, and timed P2P runtime. CUDA timing uses the device P2P kernel time. This notebook is intentionally not executed during installation.

Select backends in `BACKENDS` and geometries in `GEOMETRIES`. CUDA point-dipole comparisons use BSR(3) as the baseline. Under the current `UniformFmm` policy, cuboid CUDA comparisons retain canonical AoS because cuboid self tensors are physical and the fixed-identity BSR selector is not active.


To build:


cmake --fresh --preset notebooks
cmake --build --preset notebooks -j
cmake --install build-notebooks

In [ ]:
import gc

import cdfmm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ORDER = 6

#DEPTHS = (2, 3, 4)
DEPTHS = (4, 5)
#GRID_SIDES = (8, 10, 15, 20, 25, 30, 35, 40, 45, 50)
GRID_SIDES = (40, 50, 60, 70)

WARMUPS = 2
TIMED_EVALUATIONS = 10

SPACING = 1.0
CUBOID_FILL = 1.0

CUDA_BSR_MAX_BYTES = 20 * 1024**3


# ------------------------------------------------------------------
# Benchmark mode
# ------------------------------------------------------------------
#
# 'bsr-vs-reduced'
#     Compare BSR3 against one explicitly selected reduced kernel.
#
# 'reduced-vs-reduced'
#     Compare the two TensorDictionary CUDA executors directly:
#     R-warp vs R-target.
#
BENCHMARK_MODE = 'reduced-vs-reduced'


# Used only when BENCHMARK_MODE == 'bsr-vs-reduced'.
#
# 'target' -> global one-thread-per-target kernel
# 'warp'   -> source-parallel warp-tile kernel
BSR_COMPARISON_REDUCED_KERNEL = 'target'


BACKENDS = (
    # ('cpu-static', cdfmm.ExecutionBackend.CPU_STATIC),
    ('cuda-full', cdfmm.ExecutionBackend.CUDA_FULL),
    # ('cuda-hybrid', cdfmm.ExecutionBackend.CUDA_M2L_P2P),
)

CUDA_BACKENDS = (
    cdfmm.ExecutionBackend.CUDA_FULL,
    cdfmm.ExecutionBackend.CUDA_M2L_P2P,
)


cube = cdfmm.CuboidSize(
    CUBOID_FILL * SPACING,
    CUBOID_FILL * SPACING,
    CUBOID_FILL * SPACING,
)


GEOMETRIES = (
    (
        'point-point',
        cdfmm.SourceGeometry.POINT_DIPOLE,
        cdfmm.TargetGeometry.POINT,
        [],
        [],
    ),

    # (
    #     'cuboid-point',
    #     cdfmm.SourceGeometry.UNIFORM_CUBOID,
    #     cdfmm.TargetGeometry.POINT,
    #     [cube],
    #     [],
    # ),

    # (
    #     'cuboid-cuboid',
    #     cdfmm.SourceGeometry.UNIFORM_CUBOID,
    #     cdfmm.TargetGeometry.VOLUME_AVERAGED_CUBOID,
    #     [cube],
    #     [cube],
    # ),
)


if BENCHMARK_MODE not in (
    'bsr-vs-reduced',
    'reduced-vs-reduced',
):
    raise ValueError(
        "BENCHMARK_MODE must be "
        "'bsr-vs-reduced' or 'reduced-vs-reduced'"
    )

if BSR_COMPARISON_REDUCED_KERNEL not in (
    'target',
    'warp',
):
    raise ValueError(
        "BSR_COMPARISON_REDUCED_KERNEL must be "
        "'target' or 'warp'"
    )

In [ ]:
def cartesian_centres(side):
    grid = np.indices(
        (side, side, side),
        dtype=float,
    ).reshape(3, -1).T

    return (
        grid - (side - 1) / 2.0
    ) * SPACING


def safe_ratio(numerator, denominator):
    if denominator == 0:
        return np.nan

    return numerator / denominator


def cuda_p2p_persistent_bytes(fmm):
    stats = fmm.cuda_plan_statistics

    return sum(
        stats[name]
        for name in (
            'p2p_tensor_bytes',
            'p2p_index_bytes',
            'p2p_row_metadata_bytes',
            'p2p_leaf_metadata_bytes',
            'p2p_identity_bytes',
        )
    )


# ------------------------------------------------------------------
# Short names used in figures/tables
# ------------------------------------------------------------------

BACKEND_ABBREVIATIONS = {
    'cuda-full': 'CF',
    'cuda-hybrid': 'CH',
    'cpu-static': 'CPU',
}

GEOMETRY_ABBREVIATIONS = {
    'point-point': 'PP',
    'cuboid-point': 'CP',
    'cuboid-cuboid': 'CC',
}

VARIANT_LABELS = {
    'bsr': 'BSR',
    'r-target': 'R-target',
    'r-warp': 'R-warp',
}


def short_case_label(backend_name, geometry_name, depth):
    backend = BACKEND_ABBREVIATIONS.get(
        backend_name,
        backend_name,
    )

    geometry = GEOMETRY_ABBREVIATIONS.get(
        geometry_name,
        geometry_name,
    )

    return f'{backend} {geometry} D{depth}'


def variant_label(variant):
    return VARIANT_LABELS.get(
        variant,
        variant,
    )


# ------------------------------------------------------------------
# Case selection
# ------------------------------------------------------------------

def should_run_case(depth, side):
    if BENCHMARK_MODE == 'reduced-vs-reduced':
        # No BSR is constructed, so do not impose the BSR memory limits.
        #and not (depth == 3 and side > 40)
        return (
            not (depth == 2 and side > 35)
            and not (depth == 3 and side > 45)
        )

    # BSR comparison mode:
    #
    # depth 2 -> stop at 30^3
    # depth 3 -> stop at 40^3
    # depth 4 -> use all configured GRID_SIDES
    return (
        not (depth == 2 and side > 30)
        and not (depth == 3 and side > 40)
    )


# ------------------------------------------------------------------
# FMM construction
# ------------------------------------------------------------------

def make_fmm(
    positions,
    backend,
    geometry,
    depth,
    variant,
):
    _, backend_value = backend

    (
        _,
        source_geometry,
        target_geometry,
        source_sizes,
        target_sizes,
    ) = geometry

    options = cdfmm.UniformFmmOptions()

    options.precision = cdfmm.StaticPrecision.FLOAT32
    options.backend = backend_value
    options.expansion_order = ORDER
    options.tree.max_level = depth

    options.source_geometry = source_geometry
    options.target_geometry = target_geometry
    options.source_sizes = source_sizes
    options.target_sizes = target_sizes

    options.use_cuboid_p2m = False
    options.use_cuboid_l2p = False

    options.fixed_target_source_indices = list(
        range(len(positions))
    )

    if backend_value in CUDA_BACKENDS:
        options.cuda_p2p_bsr_max_bytes = (
            CUDA_BSR_MAX_BYTES
        )

    if variant == 'bsr':
        options.use_reduced_symmetry_p2p = False

    elif variant == 'r-target':
        options.use_reduced_symmetry_p2p = True

        if backend_value not in CUDA_BACKENDS:
            raise RuntimeError(
                'R-target is a CUDA-specific benchmark variant'
            )

        options.cuda_dictionary_target_owned = True

    elif variant == 'r-warp':
        options.use_reduced_symmetry_p2p = True

        if backend_value not in CUDA_BACKENDS:
            raise RuntimeError(
                'R-warp is a CUDA-specific benchmark variant'
            )

        options.cuda_dictionary_target_owned = False

    else:
        raise ValueError(
            f'Unknown P2P variant: {variant}'
        )

    return cdfmm.UniformFmm(
        positions,
        positions,
        options,
    )


def validate_variant(fmm, variant):
    if variant == 'bsr':
        expected = (
            cdfmm.P2PExecutionPacking.CUDA_BSR3
        )

    else:
        expected = (
            cdfmm.P2PExecutionPacking.TENSOR_DICTIONARY
        )

    if fmm.p2p_execution_packing != expected:
        raise RuntimeError(
            f'{variant} resolved '
            f'{fmm.p2p_execution_packing}; '
            f'expected {expected}'
        )


# ------------------------------------------------------------------
# Timing one FMM object
# ------------------------------------------------------------------

def benchmark_fmm(
    fmm,
    moments,
    identities,
    is_cuda,
):
    p2p_key = (
        'cuda_p2p_kernel'
        if is_cuda
        else 'p2p'
    )

    for _ in range(WARMUPS):
        fmm.evaluate(
            moments,
            target_source_indices=identities,
        )

    p2p_samples = []
    total_samples = []
    field = None

    for _ in range(TIMED_EVALUATIONS):
        field = fmm.evaluate(
            moments,
            target_source_indices=identities,
        )['H']

        p2p_samples.append(
            1e3 * fmm.last_timings[p2p_key]
        )

        total_samples.append(
            1e3 * fmm.last_timings['total']
        )

    return (
        field,
        float(np.median(p2p_samples)),
        float(np.median(total_samples)),
    )


# ------------------------------------------------------------------
# Run one variant and immediately release its CUDA plan.
# This avoids keeping both large plans resident simultaneously.
# ------------------------------------------------------------------

def run_variant(
    positions,
    moments,
    identities,
    backend,
    geometry,
    depth,
    variant,
):
    backend_value = backend[1]
    is_cuda = backend_value in CUDA_BACKENDS

    fmm = make_fmm(
        positions,
        backend,
        geometry,
        depth,
        variant,
    )

    validate_variant(
        fmm,
        variant,
    )

    (
        field,
        p2p_ms,
        total_ms,
    ) = benchmark_fmm(
        fmm,
        moments,
        identities,
        is_cuda,
    )

    static_stats = fmm.static_plan_statistics

    if is_cuda:
        p2p_bytes = cuda_p2p_persistent_bytes(
            fmm
        )
    else:
        p2p_bytes = np.nan

    if variant == 'bsr':
        stored_tensors = static_stats[
            'p2p_interactions'
        ]
    else:
        stored_tensors = static_stats[
            'p2p_unique_tensors'
        ]

    result = {
        'field': field,
        'p2p_ms': p2p_ms,
        'total_ms': total_ms,
        'p2p_bytes': p2p_bytes,
        'stored_tensors': stored_tensors,
        'p2p_interactions': static_stats[
            'p2p_interactions'
        ],
    }

    # Important for the large cases:
    # release this complete CUDA plan before building the next one.
    del fmm
    gc.collect()

    return result


# ------------------------------------------------------------------
# Complete case
# ------------------------------------------------------------------

def run_case(
    backend,
    geometry,
    side,
    depth,
):
    backend_name, backend_value = backend
    geometry_name = geometry[0]

    if (
        backend_value
        == cdfmm.ExecutionBackend.CUDA_FULL
        and not cdfmm.cuda_full_available()
    ):
        raise RuntimeError(
            'CUDA_FULL is selected but is unavailable'
        )

    if (
        backend_value
        == cdfmm.ExecutionBackend.CUDA_M2L_P2P
        and not cdfmm.cuda_m2l_p2p_available()
    ):
        raise RuntimeError(
            'CUDA_M2L_P2P is selected but is unavailable'
        )

    if (
        BENCHMARK_MODE == 'reduced-vs-reduced'
        and backend_value not in CUDA_BACKENDS
    ):
        raise RuntimeError(
            'reduced-vs-reduced mode requires a CUDA backend'
        )

    positions = cartesian_centres(side)
    particles = len(positions)

    identities = np.arange(
        particles,
        dtype=np.int32,
    )

    moments = np.random.default_rng(
        1000 + particles
    ).normal(
        size=(particles, 3)
    )

    # --------------------------------------------------------------
    # Select the two variants
    # --------------------------------------------------------------

    if BENCHMARK_MODE == 'bsr-vs-reduced':
        reference_variant = 'bsr'

        comparison_variant = (
            'r-target'
            if BSR_COMPARISON_REDUCED_KERNEL == 'target'
            else 'r-warp'
        )

    else:
        # Use warp as the reference so ratio > 1 means
        # target-owned is faster.
        reference_variant = 'r-warp'
        comparison_variant = 'r-target'

    # --------------------------------------------------------------
    # Execute sequentially
    # --------------------------------------------------------------

    reference = run_variant(
        positions,
        moments,
        identities,
        backend,
        geometry,
        depth,
        reference_variant,
    )

    comparison = run_variant(
        positions,
        moments,
        identities,
        backend,
        geometry,
        depth,
        comparison_variant,
    )

    # --------------------------------------------------------------
    # Correctness
    # --------------------------------------------------------------

    np.testing.assert_allclose(
        comparison['field'],
        reference['field'],
        rtol=5e-5,
        atol=5e-6,
    )

    difference = np.abs(
        comparison['field']
        - reference['field']
    )

    max_abs_difference = float(
        difference.max()
    )

    max_relative_difference = float(
        difference.max()
        / max(
            np.abs(reference['field']).max(),
            np.finfo(float).tiny,
        )
    )

    # --------------------------------------------------------------
    # Result
    # --------------------------------------------------------------

    return {
        'mode': BENCHMARK_MODE,

        'backend': backend_name,
        'geometry': geometry_name,

        'case': short_case_label(
            backend_name,
            geometry_name,
            depth,
        ),

        'depth': depth,
        'order': ORDER,
        'grid_side': side,

        'particles': particles,
        'particles_per_leaf': (
            particles / (8**depth)
        ),

        'reference': variant_label(
            reference_variant
        ),

        'comparison': variant_label(
            comparison_variant
        ),

        'reference_stored_tensors': (
            reference['stored_tensors']
        ),

        'comparison_stored_tensors': (
            comparison['stored_tensors']
        ),

        'reference_p2p_bytes': (
            reference['p2p_bytes']
        ),

        'comparison_p2p_bytes': (
            comparison['p2p_bytes']
        ),

        'p2p_memory_ratio': safe_ratio(
            reference['p2p_bytes'],
            comparison['p2p_bytes'],
        ),

        'p2p_ms_reference': (
            reference['p2p_ms']
        ),

        'p2p_ms_comparison': (
            comparison['p2p_ms']
        ),

        'p2p_speedup': safe_ratio(
            reference['p2p_ms'],
            comparison['p2p_ms'],
        ),

        'total_ms_reference': (
            reference['total_ms']
        ),

        'total_ms_comparison': (
            comparison['total_ms']
        ),

        'total_speedup': safe_ratio(
            reference['total_ms'],
            comparison['total_ms'],
        ),

        'max_abs_field_difference': (
            max_abs_difference
        ),

        'max_relative_field_difference': (
            max_relative_difference
        ),
    }

In [ ]:
rows = [
    run_case(
        backend,
        geometry,
        side,
        depth,
    )
    for backend in BACKENDS
    for geometry in GEOMETRIES
    for depth in DEPTHS
    for side in GRID_SIDES
    if should_run_case(depth, side)
]

results = pd.DataFrame(rows)

display(results)

In [ ]:
fig, axes = plt.subplots(
    2,
    3,
    figsize=(18, 9),
    constrained_layout=True,
)

for (
    backend,
    geometry,
    depth,
), group in results.groupby(
    [
        'backend',
        'geometry',
        'depth',
    ]
):
    group = group.sort_values(
        'particles'
    )

    case = short_case_label(
        backend,
        geometry,
        depth,
    )

    reference = group[
        'reference'
    ].iloc[0]

    comparison = group[
        'comparison'
    ].iloc[0]

    reference_label = (
        f'{case} {reference}'
    )

    comparison_label = (
        f'{case} {comparison}'
    )

    # ------------------------------------------------------------
    # Stored Tensor6 entries
    # ------------------------------------------------------------

    axes[0, 0].loglog(
        group['particles'],
        group['reference_stored_tensors'],
        'o--',
        label=reference_label,
    )

    axes[0, 0].loglog(
        group['particles'],
        group['comparison_stored_tensors'],
        'o-',
        label=comparison_label,
    )

    # ------------------------------------------------------------
    # Persistent CUDA P2P memory
    # ------------------------------------------------------------

    axes[0, 1].loglog(
        group['particles'],
        group['reference_p2p_bytes'],
        'o--',
        label=reference_label,
    )

    axes[0, 1].loglog(
        group['particles'],
        group['comparison_p2p_bytes'],
        'o-',
        label=comparison_label,
    )

    # ------------------------------------------------------------
    # Field agreement
    # ------------------------------------------------------------

    axes[0, 2].semilogx(
        group['particles'],
        group['max_relative_field_difference'],
        'o-',
        label=case,
    )

    # ------------------------------------------------------------
    # P2P runtime
    # ------------------------------------------------------------

    axes[1, 0].loglog(
        group['particles'],
        group['p2p_ms_reference'],
        'o--',
        label=reference_label,
    )

    axes[1, 0].loglog(
        group['particles'],
        group['p2p_ms_comparison'],
        'o-',
        label=comparison_label,
    )

    # ------------------------------------------------------------
    # Complete evaluation runtime
    # ------------------------------------------------------------

    axes[1, 1].loglog(
        group['particles'],
        group['total_ms_reference'],
        'o--',
        label=reference_label,
    )

    axes[1, 1].loglog(
        group['particles'],
        group['total_ms_comparison'],
        'o-',
        label=comparison_label,
    )

    # ------------------------------------------------------------
    # Complete evaluation speedup
    # ------------------------------------------------------------

    axes[1, 2].semilogx(
        group['particles'],
        group['total_speedup'],
        'o-',
        label=case,
    )


axes[0, 0].set(
    title='Stored Tensor6 entries',
    xlabel='Particles',
    ylabel='Tensor count',
)

axes[0, 1].set(
    title='Persistent CUDA P2P memory',
    xlabel='Particles',
    ylabel='Bytes',
)

axes[0, 2].set(
    title='Field agreement',
    xlabel='Particles',
    ylabel='Maximum relative field difference',
)

axes[1, 0].set(
    title='P2P runtime',
    xlabel='Particles',
    ylabel='Median P2P time [ms]',
)

axes[1, 1].set(
    title='Complete evaluation runtime',
    xlabel='Particles',
    ylabel='Median total time [ms]',
)


reference = results['reference'].iloc[0]
comparison = results['comparison'].iloc[0]

axes[1, 2].set(
    title='Complete evaluation speedup',
    xlabel='Particles',
    ylabel=f'{reference} / {comparison}',
)

axes[1, 2].axhline(
    1.0,
    linestyle='--',
    linewidth=1,
)


for axis in axes.flat:
    axis.grid(
        True,
        which='both',
        alpha=0.25,
    )

    axis.legend(
        fontsize=7,
        ncol=2,
    )


plt.show()


display(
    results[
        [
            'backend',
            'geometry',
            'depth',
            'particles',
            'particles_per_leaf',

            'reference',
            'comparison',

            'reference_p2p_bytes',
            'comparison_p2p_bytes',
            'p2p_memory_ratio',

            'p2p_ms_reference',
            'p2p_ms_comparison',
            'p2p_speedup',

            'total_ms_reference',
            'total_ms_comparison',
            'total_speedup',

            'max_relative_field_difference',
        ]
    ]
)